In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/Hydrology_Applications/Climate_change_analysis

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/Hydrology_Applications/Climate_change_analysis


In [ ]:
!pip install xarray rioxarray geopandas pandas numpy matplotlib

In [ ]:
import geopandas as gpd
import xarray as xr
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import geopandas as gpd
import xarray as xr
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# =============================================================
# 1. FILE PATHS
# =============================================================
# CMIP6 NetCDF file (Daily precipitation)
# Variable: pr
# Units: kg m-2 s-1

# -------------------------------------------------
# File Paths
# -------------------------------------------------
GCM = 'ACCESS-CM2'
period = 'historical'
params = 'pr'
data_path = "/content/drive/MyDrive/NEX_GDDP_CMIP6/ACCESS-CM2/historical/pr/"
print(data_path)

/content/drive/MyDrive/NEX_GDDP_CMIP6/ACCESS-CM2/historical/pr/


In [ ]:
# =============================================================
# 2. READ DATA
# =============================================================
# -------------------------------------------------
ds = xr.open_mfdataset(
    os.path.join(data_path, "*.nc"),
    combine="by_coords"
)

# -------------------------------------------------
# 4.2 Assign Coordinate Reference System (CRS)
# -------------------------------------------------
ds = ds.rio.write_crs("EPSG:4326")  # WGS84


In [ ]:
shp_path  = r"Basin_shp/Karnali_basin.shp"

# -------------------------------------------------
# 4.3 Read Basin Shapefile
# -------------------------------------------------
basin = gpd.read_file(shp_path)
basin = basin.to_crs("EPSG:4326")   # Ensure CRS consistency

# -------------------------------------------------
# 4.4 Clip Dataset to Basin Boundary
# -------------------------------------------------
ds_clipped = ds.rio.clip(
    basin.geometry,
    basin.crs,
    drop=True
)
ds_clipped


# Extract precipitation variable
# Original units: kg m-2 s-1
pr = ds_clipped["pr"]
pr

<xarray.DataArray 'pr' (time: 10957, lat: 9, lon: 13)> Size: 5MB
dask.array<getitem, shape=(10957, 9, 13), dtype=float32, chunksize=(1, 9, 13), chunktype=numpy.ndarray>
Coordinates:
  * time         (time) datetime64[ns] 88kB 1985-01-01T12:00:00 ... 2014-12-3...
  * lat          (lat) float64 72B 28.38 28.62 28.88 29.12 ... 29.88 30.12 30.38
  * lon          (lon) float64 104B 80.62 80.88 81.12 ... 83.12 83.38 83.62
    spatial_ref  int64 8B 0
Attributes:
    standard_name:  precipitation_flux
    long_name:      Precipitation
    units:          kg m-2 s-1
    comment:        includes both liquid and solid phases
    cell_methods:   area: time: mean
    cell_measures:  area: areacella

In [ ]:
# Excel file containing station metadata
# Required columns:
# - Index No.
# - Lat
# - Lon
station_file = r"Karnali_1986_2015.xlsx"

# Output directory (model-specific folder)
out_dir = "Stationwise/" + GCM  + '/' + period

os.makedirs(out_dir, exist_ok=True)

# Final output CSV path
out_csv = f"{out_dir}/{GCM}_{period}_karnali_pr.csv"


# Read station coordinates
stations = pd.read_excel(station_file, sheet_name="Sheet1")
stations

In [ ]:
# =============================================================
# 2A. VISUAL VALIDATION (SPATIAL CHECK)
# =============================================================

# Read Karnali Basin shapefile
shp_path = r"Basin_shp/Karnali_basin.shp"
basin = gpd.read_file(shp_path)

# Ensure shapefile is in WGS84 geographic coordinates
# (CMIP6 data is in lat/lon EPSG:4326)
if basin.crs != "EPSG:4326":
    basin = basin.to_crs("EPSG:4326")

# Select first day from dataset (time index = 0)
# Convert to mm/day for interpretation
day1 = pr.isel(time=0) * 86400

# Create spatial validation plot
fig, ax = plt.subplots(figsize=(8,6))

# Plot precipitation grid (raster-like visualization)
mesh = ax.pcolormesh(
    day1.lon,
    day1.lat,
    day1,
    shading="auto"
)

# Plot basin boundary
basin.boundary.plot(ax=ax)

# Plot station locations
ax.scatter(
    stations["Lon"],
    stations["Lat"],
)

ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("ACCESS-CM2 Precipitation (mm/day) - Day 1")

plt.colorbar(mesh, ax=ax, label="Precipitation (mm/day)")
plt.show()

In [ ]:
# =============================================================
# 3. VECTORIZED GRID-CELL TIME SERIES EXTRACTION
# =============================================================

from tqdm.auto import tqdm
import xarray as xr
import pandas as pd
import numpy as np

print("Extracting nearest grid cells...")

# -------------------------------------------------------------
# Create DataArray of station coordinates
# -------------------------------------------------------------
station_ids = stations["Index No."].astype(str).values

station_lats = xr.DataArray(
    stations["Lat"].values,
    dims="station",
    coords={"station": station_ids}
)

station_lons = xr.DataArray(
    stations["Lon"].values,
    dims="station",
    coords={"station": station_ids}
)

# -------------------------------------------------------------
# Extract ALL stations simultaneously
# -------------------------------------------------------------
station_pr = pr.sel(
    lat=station_lats,
    lon=station_lons,
    method="nearest"
)

print("Extraction completed!")

# =============================================================
# 4. CONVERT TO mm/day
# =============================================================

station_pr = station_pr * 86400

# =============================================================
# 5. CREATE DATAFRAME
# =============================================================

print("Creating dataframe...")

df_raw_gcm = pd.DataFrame(
    station_pr.values,
    index=station_pr.time.values,
    columns=station_ids
)

df_raw_gcm.index.name = "Date"
df_raw_gcm.reset_index(inplace=True)

# =============================================================
# 6. FORMAT DATE
# =============================================================

df_raw_gcm["Date"] = pd.to_datetime(df_raw_gcm["Date"]).dt.normalize()

# =============================================================
# 7. RENAME COLUMNS WITH GRID COORDINATES
# =============================================================

new_columns = {"Date": "Date"}

for i, station in enumerate(station_ids):

    grid_lat = float(station_pr.lat.values[i])
    grid_lon = float(station_pr.lon.values[i])

    new_columns[station] = f"{station} ({grid_lat:.3f}, {grid_lon:.3f})"

df_raw_gcm.rename(columns=new_columns, inplace=True)

Extracting nearest grid cells...
Extraction completed!
Creating dataframe...


In [ ]:
# =============================================================
# 8. SAVE CSV
# =============================================================

df_raw_gcm.to_csv(out_csv, index=False)

print("======================================")
print("Finished Successfully")
print(f"Stations extracted : {len(station_ids)}")
print(f"Timesteps          : {len(df_raw_gcm)}")
print(f"Output             : {out_csv}")
print("======================================")

Finished Successfully
Stations extracted : 10
Timesteps          : 10957
Output             : Stationwise/ACCESS-CM2/historical/ACCESS-CM2_historical_karnali_pr.csv
